# Vantage — first run on real hardware (Colab T4)

Every Vantage result so far came from its simulator. This run replaces them with measurements:
a real LLM server on a real GPU, driven by **Vectora's real workload** (its own questions, retrieved
context and exact prompt, exported by `scripts/export_vectora.py`).

| # | Question | Sweep |
|---|---|---|
| 1 | How many concurrent users can one T4 serve before latency breaks the SLO? | `t4-concurrency` (1→32 users) |
| 2 | Does quantisation pay for itself, and does it hurt answers? | `t4-quantization` (fp16 vs q8 vs q4, quality-gated) |
| 3 | At what request rate does the server fall over? | `t4-offered-load` (Poisson arrivals) |
| 4 | Is a semantic cache safe for this workload? | `cache-sim` (hit rate vs false-hit rate) |
| 5 | *(optional)* Does vLLM's continuous batching beat Ollama? | `t4-vllm-concurrency` |

**To start:** Runtime → Change runtime type → **T4 GPU**. Run cell 1 (approve Drive), cell 2 (check `100% GPU`), then click cell 3 and **Runtime → Run cell and below**.
Results land in `MyDrive/vantage_results/<run>/` as each sweep finishes. Expect ~1.5–2.5 h; the vLLM part adds ~30–45 min.

## 1. Drive

In [ ]:
import datetime, os
RUN = "t4_" + datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
from google.colab import drive
drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/vantage_results/{RUN}"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

## 2. Ollama on the GPU, three precisions of one model

- `OLLAMA_CONTEXT_LENGTH=8192`: Vectora's prompts are ~2k tokens with the system prompt. Ollama's default window can truncate them **silently**, which would make every number wrong.
- `OLLAMA_NUM_PARALLEL=16`: 16 request slots, so concurrency above 16 queues — the concurrency sweep should show that knee.
- `pciutils lshw`: without them Ollama's installer cannot see the GPU and installs a CPU build (Vectora run 1's problem).

In [ ]:
MODELS = {"fp16": "qwen2.5:3b-instruct-fp16", "q8": "qwen2.5:3b-instruct-q8_0", "q4": "qwen2.5:3b-instruct-q4_K_M"}
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, urllib.request
OLLAMA_ENV = {**os.environ, "OLLAMA_KEEP_ALIVE": "-1", "OLLAMA_NUM_PARALLEL": "16",
              "OLLAMA_CONTEXT_LENGTH": "8192", "OLLAMA_MAX_LOADED_MODELS": "1"}
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"),
                          stderr=subprocess.STDOUT, env=OLLAMA_ENV)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2); print("ollama up"); break
    except Exception:
        time.sleep(1)
for m in MODELS.values():
    !ollama pull {m} 2>&1 | tail -1
!ollama run {MODELS["fp16"]} "hi" > /dev/null 2>&1; ollama ps

## 3. Vantage, and the Vectora workload

In [ ]:
!git clone -q https://github.com/SiddharthaRaja/vantage.git /content/vantage
%cd /content/vantage
!git log --oneline -3 | tee {OUT}/commit.txt
!pip install -q -e ".[dev]" sentence-transformers 2>&1 | grep -v -i "warning\|dependency resolver\|incompatible" | tail -2
!test -f data/workload.vectora.jsonl && echo "workload present" || echo "MISSING data/workload.vectora.jsonl - run scripts/export_vectora.py and push first"
!vantage workload data/workload.vectora.jsonl 2>&1 | tee {OUT}/workload_shape.txt
!vantage doctor -b ollama.qwen25-3b-fp16 -w data/workload.vectora.jsonl 2>&1 | tail -5

## 4. Sanity check: is the whole prompt reaching the model?

Sends one real Vectora prompt and prints how many prompt tokens the server says it read. It should be well above 1,000. If it is suspiciously close to a round number like 2048 or 4096, the prompt is being truncated and the sweeps would be meaningless.

In [ ]:
import json
from openai import OpenAI
row = json.loads(open("data/workload.vectora.jsonl").readline())
system = open("data/workload.vectora.system.txt").read()
client = OpenAI(base_url="http://127.0.0.1:11434/v1", api_key="ollama")
t0 = time.time()
r = client.chat.completions.create(model=MODELS["fp16"], max_tokens=192, temperature=0,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": row["prompt"]}])
print(f"prompt tokens read by server: {r.usage.prompt_tokens}   answer tokens: {r.usage.completion_tokens}   {time.time()-t0:.1f}s")
print("question:", row["question"]); print("answer  :", r.choices[0].message.content[:500])
print("gold    :", row["reference_answer"])
open(f"{OUT}/sanity_check.txt", "w").write(f"prompt_tokens={r.usage.prompt_tokens}\n{r.choices[0].message.content}\n")

## Helper

Runs a sweep, logs it to Drive, copies the run folder (raw data, charts, `REPORT.md`) to Drive.

In [ ]:
import shutil, glob
def sweep(name, extra=""):
    log = f"{OUT}/{name}.log"
    t0 = time.time()
    with open(log, "w") as f:
        subprocess.run(f"vantage sweep -c configs/sweeps/{name}.yaml {extra}", shell=True,
                       stdout=f, stderr=subprocess.STDOUT)
    runs = sorted(glob.glob(f"results/{name}/*"))
    if runs:
        shutil.copytree(runs[-1], f"{OUT}/{name}", dirs_exist_ok=True)
    print(f"[{name}] {time.time()-t0:.0f}s -> {runs[-1] if runs else 'NO RESULTS (see log)'}")
    print(open(log).read()[-1500:])

## 5. Question 1 — concurrency (fp16)

In [ ]:
sweep("t4-concurrency")

## 6. Question 2 — quantisation, quality-gated

In [ ]:
sweep("t4-quantization")

## 7. Question 3 — offered load (q4)

In [ ]:
sweep("t4-offered-load")

## 8. Question 4 — is a semantic cache safe here?

Replays Vectora's questions (with paraphrases) through a cache at each similarity threshold. A *false hit* serves a cached answer to a different question. Title-free QASPER questions like "What are the baselines?" are identical across papers but have different answers, so expect this to be a sharp result.

In [ ]:
os.environ["VANTAGE_EMBEDDING_MODEL"] = "sentence-transformers/all-MiniLM-L6-v2"
!vantage cache-sim -w data/workload.vectora.jsonl --embedder st --repeats 5 --out {OUT}/cache_sim.json 2>&1 | tee {OUT}/cache_sim.log | tail -20

## 9. *(optional)* vLLM, the production server

Stops Ollama (it holds the GPU), installs vLLM and runs the identical concurrency sweep. If vLLM does not start on the T4 within 10 minutes, this section is skipped and everything above is unaffected.

In [ ]:
ollama.terminate(); time.sleep(5)
!pkill -f "ollama" ; sleep 3; nvidia-smi --query-gpu=memory.used --format=csv
!pip install -q vllm 2>&1 | tail -1
vlog = open("/content/vllm.log", "w")
vllm = subprocess.Popen(["vllm", "serve", "Qwen/Qwen2.5-3B-Instruct", "--dtype", "half", "--max-model-len", "8192",
                         "--gpu-memory-utilization", "0.90", "--enable-prefix-caching", "--port", "8000"],
                        stdout=vlog, stderr=subprocess.STDOUT)
VLLM_UP = False
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2); VLLM_UP = True; break
    except Exception:
        if vllm.poll() is not None: break
        time.sleep(5)
print("vLLM up" if VLLM_UP else "vLLM did not start - see /content/vllm.log (copied to Drive); skipping")
shutil.copy("/content/vllm.log", f"{OUT}/vllm_server.log")
if VLLM_UP:
    sweep("t4-vllm-concurrency")
    vllm.terminate()

## 10. Summary

Collects every `REPORT.md` into one `SUMMARY.md` on Drive. In the morning, download `MyDrive/vantage_results/<run>` and send it to me.

In [ ]:
parts = [f"# Vantage on a Colab T4 — {RUN}", ""]
for name in ["t4-concurrency", "t4-quantization", "t4-offered-load", "t4-vllm-concurrency"]:
    rep = f"{OUT}/{name}/REPORT.md"
    if os.path.exists(rep):
        parts += [f"\n---\n\n# {name}\n", open(rep).read()]
if os.path.exists(f"{OUT}/cache_sim.log"):
    parts += ["\n---\n\n# cache-sim\n", "```", open(f"{OUT}/cache_sim.log").read()[-3000:], "```"]
open(f"{OUT}/SUMMARY.md", "w").write("\n".join(parts))
!cp /content/ollama.log {OUT}/ 2>/dev/null; date | tee {OUT}/finished_at.txt
!ls -la {OUT}